# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to access, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library and Python. All objects are referenced by their Croissant `@id` fields, supporting reproducibility and machine-readability.

### Dataset Source
The dataset is described using a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()
print("Dataset Title:", metadata_json.get('name', 'N/A'))
print("Description:", metadata_json.get('description', 'N/A'))

## 2. Data Overview
Review available record sets and the fields/columns defined in the Croissant schema with their `@id`s.

In [ ]:
# Get the list of record sets defined by their @id
record_sets = [rec['@id'] for rec in metadata_json.get('recordSet', [])]
print("Available record sets (by @id):")
for rid in record_sets:
    print(" -", rid)

if not record_sets:
    # Try heuristic: infer record set from distributions (if recordSet missing)
    distributions = metadata_json.get('distribution', [])
    print("No explicit 'recordSet' field found. Attempting to infer from distributions:")
    for dist in distributions:
        print(" -", dist['@id'])

# List available fields/columns (usually in 'field' or 'column' under each recordSet)
print("\nFields/columns by record set:")
for rec in metadata_json.get('recordSet', []):
    rid = rec.get('@id')
    # Croissant recordSet may list 'field', 'column', or both
    fields = rec.get('field', [])
    columns = rec.get('column', [])
    if fields:
        fnames = [f['@id'] if isinstance(f, dict) and '@id' in f else str(f) for f in fields]
        print(f"- RecordSet {rid}: Fields: {fnames}")
    if columns:
        cnames = [c['@id'] if isinstance(c, dict) and '@id' in c else str(c) for c in columns]
        print(f"- RecordSet {rid}: Columns: {cnames}")

## 3. Data Extraction
Load data from a specific record set (by `@id`) into a DataFrame for further analysis. 
Below, we use the main tabular data which is likely to be the dataset's primary CSV, referenced by the `@id` from 'distribution'.

In [ ]:
# Choose record set @id for extraction. Fallback: use distribution @id as main table if recordSet is empty.
if record_sets:
    main_record_set_id = record_sets[0]  # Assuming first is the main record set
else:
    # Fallback if no explicit record sets (as appears in this schema)
    main_record_set_id = metadata_json['distribution'][0]['@id']

print(f"Extracting records from record set: {main_record_set_id}")

records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print("DataFrame columns:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Filter records, normalize numeric fields, and group/categorize the data for initial exploration.

All field and grouping operations use the field's `@id` from the schema or the DataFrame columns as they appear after loading.

In [ ]:
# Find a numeric column: try possible candidates by common field names.
import numpy as np
# Candidates from schema description (e.g., normalized abundance, counts, molecular_weight, etc.)
possible_numeric_fields = [
    'coverage_percent', 'peptide_count', 'molecular_weight', 'normalized_abundance',
    'peptide_counts', 'MW', 'MW (Da)', 'abundance', 'Score', 'pI', 'MW_Da' # typical proteomics columns
]
numeric_field = None
for col in df.columns:
    if any(k.lower().replace(' ','_').replace('(','').replace(')','') in col.lower().replace(' ','_').replace('(','').replace(')','') for k in possible_numeric_fields):
        try:
            # Test if column is numeric
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if df[col].notnull().sum() > 0:
                numeric_field = col
                break
        except Exception as e:
            continue

if numeric_field is None:
    numeric_field = df.select_dtypes(include=[np.number]).columns[0]
    print(f"No candidate found; defaulting to: {numeric_field}")
else:
    print(f"Using numeric field for filtering: {numeric_field}")

# Example threshold (pick a reasonable cutoff for peptide count/molecular weight/abundance)
threshold = np.nanpercentile(df[numeric_field], 75)
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records where {numeric_field} > {threshold} (top quartile): {len(filtered_df)} rows")
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a possible categorical column (e.g., 'accession', 'Group', etc.)
possible_group_fields = ['accession', 'sample', 'modification', 'attribute', 'description', 'group']
group_field = None
for col in df.columns:
    if any(k.lower() in col.lower() for k in possible_group_fields):
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print('No suitable group field found for grouping.')

## 5. Visualization
Display key distributions, e.g., histogram of numeric field and a bar plot of mean per group (if grouping is possible).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If grouped, show barplot of group means (top groups only)
if group_field:
    top_groups = grouped_df.sort_values(ascending=False).head(10)
    plt.figure(figsize=(10,4))
    sns.barplot(x=top_groups.index.astype(str), y=top_groups.values)
    plt.title(f"Top 10 groups by mean {numeric_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we have loaded and explored the FAIR^2 dataset schema describing extracellular vesicle mass spectrometry data, retrieved actual records, performed basic normalization and grouping, and visualized distributions. All data operations referenced entities by their `@id` when possible, following FAIR, reproducible standards. For deeper analysis (e.g., specific hypotheses or machine learning), further domain-specific cleaning and rich domain metadata integration would be advised.